# Tucker / HOSVD 基礎実装

このNotebookでは、**TensorLyを使う前に、PyTorchだけでHOSVD/Tucker分解の基本処理を自作する**。

目的は、各modeの展開・factor matrix・core tensor・再構成の関係を、自分で実装して確認すること。

自作後は、TensorLyに用意されている対応APIの使い方も確認し、再構成誤差やshapeの整合を比較する。

> TensorLyをPyTorch Tensorのまま使う場合は `tensorly.set_backend("pytorch")` を指定できる。


## 1. 分解対象テンソル


In [ ]:
import torch

X = torch.tensor(
    [
        [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]],
        [[2.0, 1.0], [4.0, 3.0], [6.0, 5.0], [8.0, 7.0]],
        [[1.0, 3.0], [2.0, 4.0], [3.0, 5.0], [4.0, 6.0]],
    ],
    dtype=torch.float32,
)

print(X)
print("shape:", X.shape)


## 2. mode-n unfolding


In [ ]:
def unfold(X, mode):
    """
    目的: 指定したmodeを基準に、テンソルを2次元行列へ展開する。

    X: 展開するテンソル
    mode: 展開の基準にする軸番号
    """
    pass


### ライブラリでは

TensorLyには同じ目的の `tensorly.base.unfold` が用意されている。

```python
import tensorly as tl
tl.set_backend("pytorch")

X_mode0 = tl.unfold(X, mode=0)
```

PyTorchには **mode-n unfolding専用の同等APIはない**。なお、`torch.Tensor.unfold` はスライディングウィンドウを取り出す別の処理なので、ここでいうunfoldingとは異なる。


## 3. mode-n product


In [ ]:
def mode_dot(X, matrix, mode):
    """
    目的: テンソルの指定したmodeに2次元行列を作用させる。

    X: mode積を行うテンソル
    matrix: 指定したmodeに作用させる2次元行列
    mode: 行列を作用させる軸番号
    """
    pass


### ライブラリでは

TensorLyには `tensorly.tenalg.mode_dot` が用意されている。

```python
from tensorly.tenalg import mode_dot

Y = mode_dot(X, matrix, mode=0)
```

PyTorchにはn-mode product専用の同等APIはない。一般的なテンソル縮約には `torch.tensordot` があるが、`mode_dot` の置き換えとしてそのまま同じ引数で使う関数ではない。


## 4. HOSVD


In [ ]:
def hosvd(X, ranks):
    """
    目的: 指定したmodeごとに低rank近似を行い、Tucker分解に必要なcore tensorとfactor matrixを求める。

    X: Tucker分解するテンソル
    ranks: {mode: rank} 形式で、分解対象の軸とそのrankを指定する辞書
    """
    pass


### ライブラリでは

PyTorchにはHOSVDそのものを行う関数はなく、行列SVDの `torch.linalg.svd` が自作時の基本部品になる。

```python
U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
```

TensorLyにはTucker分解として `tensorly.decomposition.tucker`、指定modeだけを分解する `partial_tucker` がある。

```python
from tensorly.decomposition import tucker, partial_tucker

# 全modeをTucker分解
core_tl, factors_tl = tucker(
    X,
    rank=[2, 2, 2],
    init="svd",
)

# mode 0, 1だけをTucker分解
core_partial, factors_partial = partial_tucker(
    X,
    rank=[2, 2],
    modes=[0, 1],
    init="svd",
)
```

**注意:** TensorLyの `tucker` / `partial_tucker` はHOSVDそのものではなく、SVDなどを初期値にした **Higher Order Orthogonal Iteration (HOI/HOOI)** によるTucker分解。したがって、自作HOSVDとの比較ではfactorの値そのものの完全一致ではなく、shape・再構成結果・再構成誤差を確認する。


## 5. Tucker再構成


In [ ]:
def reconstruct_tucker(core, factors):
    """
    目的: core tensorとfactor matrixから、元テンソルの近似を再構成する。

    core: Tucker分解で得られたcore tensor
    factors: 各modeに対応するfactor matrix
    """
    pass


### ライブラリでは

TensorLyには `tensorly.tucker_tensor.tucker_to_tensor` が用意されている。

```python
from tensorly.tucker_tensor import tucker_to_tensor

X_reconstructed = tucker_to_tensor((core, factors))
```

PyTorchにはTucker形式 `(core, factors)` を直接再構成する専用APIはない。


## 6. 確認

自作したHOSVDで分解・再構成し、shapeと再構成誤差を確認する。


## 7. TensorLyとの比較

自作実装が完成したら、上で確認したTensorLyのAPIを使って同じテンソルを処理し、以下を比較する。

- core tensor のshape
- factor matrix のshape
- 再構成後のテンソルのshape
- 再構成誤差

TensorLyのTucker分解はHOI/HOOIを使うため、自作HOSVDとfactorの数値が完全一致すること自体は比較目標にしない。
